In [ ]:
!pip install datasets transformers evaluate scikit-learn joblib --quiet

from datasets import load_dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from sklearn.preprocessing import LabelEncoder
import numpy as np, evaluate, joblib, time

In [ ]:
# Load the PlantVillage image-text pairs dataset
ds = load_dataset("ButterChicken98/plantvillage-image-text-pairs", split="train")

print("✅ Dataset loaded:", len(ds))
print("📋 Columns:", ds.column_names)
print("🔹 Example row:", ds[0])

✅ Dataset loaded: 20638
📋 Columns: ['image', 'caption', 'captions']
🔹 Example row: {'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x796DF21DBC50>, 'caption': 'Tomato healthy', 'captions': ['A vibrant green and healthy tomato leaf with smooth, spotless surface.', 'A healthy Solanum lycopersicum leaf, free of disease or pests, with a uniform green color.', 'A fresh tomato leaf outdoors, glowing in sunlight with a smooth and unblemished surface.', 'A clean and healthy tomato leaf image, perfect for comparison in plant health datasets.']}


In [ ]:
# ✅ Correct column names
text_col  = "caption" # Changed from "text"
label_col = "caption" # Changed from "plant_name"

# Encode string labels into numbers
le = LabelEncoder()
labels = le.fit_transform(ds[label_col])
ds = ds.add_column("encoded_label", labels)
num_labels = len(le.classes_)
print("✅ Encoded", num_labels, "unique labels")

# ✅ FAST MODE — only 100 samples for 2–3 minute quick test
ds_small = ds.select(range(100))
ds_split = ds_small.train_test_split(test_size=0.2)

✅ Encoded 15 unique labels


In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(batch[text_col], truncation=True, padding="max_length", max_length=128)

ds_tokenized = ds_split.map(tokenize, batched=True)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
)

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)

training_args = TrainingArguments(
    output_dir="./models/text_quick",
    eval_strategy="epoch",
    save_strategy="no",
    num_train_epochs=2,            # only 2 epochs = fast
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=3e-5,
    logging_dir="./logs",
    log_level="error"
)

# Rename the 'encoded_label' column to 'labels'
ds_tokenized["train"] = ds_tokenized["train"].rename_column("encoded_label", "labels")
ds_tokenized["test"] = ds_tokenized["test"].rename_column("encoded_label", "labels")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_tokenized["train"],
    eval_dataset=ds_tokenized["test"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
# 5️⃣ Train (2–3 minutes)
import os
os.environ["WANDB_DISABLED"] = "true"   # 🚫 Turn off W&B logging permanently

start = time.time()
trainer.train()
print(f"✅ Training finished in {(time.time()-start)/60:.2f} min")

trainer.save_model("./models/text_quick")
joblib.dump(le, "./models/label_encoder.joblib")
print("💾 Model & encoder saved!")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 2.4927401542663574, 'eval_accuracy': 0.25, 'eval_runtime': 3.5418, 'eval_samples_per_second': 5.647, 'eval_steps_per_second': 0.847, 'epoch': 1.0}


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 2.380516767501831, 'eval_accuracy': 0.2, 'eval_runtime': 4.3915, 'eval_samples_per_second': 4.554, 'eval_steps_per_second': 0.683, 'epoch': 2.0}
{'train_runtime': 121.9742, 'train_samples_per_second': 1.312, 'train_steps_per_second': 0.164, 'train_loss': 2.537051200866699, 'epoch': 2.0}
✅ Training finished in 2.04 min
💾 Model & encoder saved!


In [ ]:
from transformers import pipeline

# Load model and tokenizer
nlp = pipeline("text-classification", model="./models/text_quick", tokenizer="distilbert-base-uncased")
le  = joblib.load("./models/label_encoder.joblib")

# Example symptom description
sample = "yellow patches and powdery coating on tomato leaves"
pred = nlp(sample, top_k=None)[0]

label_idx = int(pred["label"].replace("LABEL_", ""))
disease = le.inverse_transform([label_idx])[0]

print(f"🪴 Predicted Disease: {disease} ({pred['score']*100:.1f}%)")

🪴 Predicted Disease: Tomato YellowLeaf Curl Virus (10.5%)
